# Example repro of Geometry of Truth

In [1]:
import shlex
import subprocess
import sys

from pathlib import Path

In [2]:
REPO_ROOT   = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS_DIR = REPO_ROOT / "results"

MODELS = [
    "meta-llama/Llama-2-13b-hf",
    "google/gemma-4-26B-A4B-it",
]

def slug(model_id: str) -> str:
    return model_id.replace("/", "__")

def out_dir(model_id: str) -> Path:
    return RESULTS_DIR / slug(model_id)


In [3]:
def run_model(model_id: str, force: bool = False) -> None:
    od = out_dir(model_id)
    od.mkdir(parents=True, exist_ok=True)

    if not force and (od / "causal.json").exists():
        print(f"[orchestrator] {model_id}: already complete, skipping")
        return

    cmd = [sys.executable, "-m", "probelab.got.pipeline", model_id, "--out", str(od)]
    log_path = od / "run.log"
    print(f"[orchestrator] {model_id}: launching (log -> {log_path})\n  {shlex.join(cmd)}\n", flush=True)

    with log_path.open("w") as logf:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            logf.write(line); logf.flush(); print(line, end="")
        rc = proc.wait()

    if rc != 0:
        raise RuntimeError(f"{model_id} pipeline failed (exit code {rc}). Log: {log_path}")

    print(f"\n[orchestrator] {model_id}: done\n")


In [4]:
from probelab.got import (
    load_runs,
    plot_predictive_vs_causal, plot_probe_direction_stability,
    plot_best_layers, plot_overfitting,
    plot_probe_direction_vs_best_causal
)

runs = load_runs(RESULTS_DIR, MODELS)

plot_predictive_vs_causal(runs).show()
plot_probe_direction_stability(runs).show()
plot_probe_direction_vs_best_causal(runs).show()
plot_best_layers(runs).show()
plot_overfitting(runs).show()